# Contract Net Protocol (CNP) | Multi-Agent Collaboration

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Dict, List
from typing_extensions import NotRequired
import json
import re
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
# Contractor capabilities
CONTRACTORS = {
    "legal_analyst": {"specialty": "legal documents, contracts, compliance", "rate": 0.8},
    "financial_analyst": {"specialty": "financial reports, earnings, budgets", "rate": 0.7},
    "general_analyst": {"specialty": "general documents, summaries, reports", "rate": 0.5},
}

class CNPState(TypedDict):
    task_announcement: str
    bids: NotRequired[Dict[str, dict]]
    awarded_to: NotRequired[str]
    contract_result: NotRequired[str]
    manager_report: NotRequired[str]

In [5]:
def announce_task(state: CNPState) -> dict:
    """Manager announces task and collects bids from contractors."""
    bids = {}
    for name, profile in CONTRACTORS.items():
        response = model.invoke(
            f"You are contractor '{name}', specializing in: {profile['specialty']}.\n"
            f"Your hourly rate factor: {profile['rate']}\n\n"
            f"Task announcement:\n{state['task_announcement']}\n\n"
            f"Submit a bid with:\n"
            f"1. Can you handle this? (yes/no)\n"
            f"2. Your suitability score (0.0-1.0)\n"
            f"3. Estimated quality you can deliver (0.0-1.0)\n"
            f"4. Brief proposal\n\n"
            f"Return JSON: {{\"can_handle\": true, \"suitability\": 0.9, \"quality\": 0.85, \"proposal\": \"...\"}}"
        )
        cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", response.content).strip()
        try:
            bid = json.loads(cleaned)
            bid["rate"] = profile["rate"]
            bids[name] = bid
        except json.JSONDecodeError:
            bids[name] = {"can_handle": False, "suitability": 0.0, "quality": 0.0, "proposal": "Parse error", "rate": profile["rate"]}
    return {"bids": bids}

def award_contract(state: CNPState) -> dict:
    """Manager evaluates bids and awards the contract."""
    bids = state.get("bids", {})
    best_score = -1.0
    best_contractor = None
    for name, bid in bids.items():
        if bid.get("can_handle", False):
            # Score = suitability * quality / rate (prefer high quality, low cost)
            score = bid.get("suitability", 0) * bid.get("quality", 0) / max(bid.get("rate", 1), 0.1)
            if score > best_score:
                best_score = score
                best_contractor = name
    if not best_contractor:
        best_contractor = "general_analyst"  # fallback
    return {"awarded_to": best_contractor}

def execute_contract(state: CNPState) -> dict:
    """Winning contractor executes the task."""
    contractor = state["awarded_to"]
    profile = CONTRACTORS[contractor]
    bid = state.get("bids", {}).get(contractor, {})
    response = model.invoke(
        f"You are '{contractor}', specializing in {profile['specialty']}.\n"
        f"You won the contract with this proposal: {bid.get('proposal', 'N/A')}\n\n"
        f"Now execute the task:\n{state['task_announcement']}\n\n"
        f"Deliver a thorough, professional result."
    )
    return {"contract_result": response.content}

def manager_review(state: CNPState) -> dict:
    """Manager reviews the contractor's deliverable."""
    response = model.invoke(
        f"You are the manager. Review this deliverable from contractor '{state['awarded_to']}'.\n\n"
        f"Original task: {state['task_announcement']}\n\n"
        f"Deliverable:\n{state['contract_result']}\n\n"
        f"Provide: acceptance decision, quality assessment, and any notes."
    )
    return {"manager_report": response.content}

In [6]:
graph = StateGraph(CNPState)
graph.add_sequence([("announce", announce_task), ("award", award_contract), ("execute", execute_contract), ("review", manager_review)])
graph.add_edge(START, "announce")
graph.add_edge("review", END)

cnp = graph.compile()

In [7]:
# Plot the workflow
plot_mermaid(cnp)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	announce(announce)
	award(award)
	execute(execute)
	review(review)
	__end__([<p>__end__</p>]):::last
	__start__ --> announce;
	announce --> award;
	award --> execute;
	execute --> review;
	review --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
result = cnp.invoke({
    "task_announcement": "Analyze the following contract clause for potential legal risks: "
                         "'The vendor shall indemnify the client for all damages arising from "
                         "the vendor's negligence, including consequential and indirect damages, "
                         "with no cap on liability.'"
})
print(f"Awarded to: {result['awarded_to']}")
print(f"\nResult:\n{result['contract_result']}")
print(f"\nManager Review:\n{result['manager_report']}")

Awarded to: general_analyst

Result:
**Analysis of Contract Clause for Potential Legal Risks**

**Clause Overview:**
The contract clause under review states: "The vendor shall indemnify the client for all damages arising from the vendor's negligence, including consequential and indirect damages, with no cap on liability."

**Potential Legal Risks:**

1. **Unlimited Liability Exposure:**
   - **Risk:** The absence of a cap on liability exposes the vendor to unlimited financial obligations. This means that the vendor could potentially face financial liabilities that far exceed the value of the contract or even their financial capacity, risking significant financial distress or insolvency.
   - **Implication:** Vendors without adequate financial reserves or insurance coverage might not sustain such extensive liabilities, leading to business instability.

2. **Broad Scope of Damages:**
   - **Risk:** By including consequential and indirect damages, the clause significantly broadens the sco

In [9]:
stream_invoke(cnp, {
    "task_announcement": "Analyze the following contract clause for potential legal risks: "
                         "'The vendor shall indemnify the client for all damages arising from "
                         "the vendor's negligence, including consequential and indirect damages, "
                         "with no cap on liability.'"
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'task_announcement': "Analyze the following contract clause for potential legal risks: 'The vendor shall indemnify the client for all damages arising from the vendor's negligence, including consequential and indirect damages, with no cap on liability.'",
 'bids': {'legal_analyst': {'can_handle': True,
   'suitability': 0.9,
   'quality': 0.85,
   'proposal': "I can assess this clause and identify potential legal risks. The clause imposes an uncapped indemnification obligation on the vendor, covering not only direct but also consequential and indirect damages. This could result in significant financial exposure for the vendor, and might discourage some vendors from engaging under these terms. Additionally, enforceability issues could arise, especially concerning consequential damages, which are often contentious. My analysis will focus on potential risk areas and suggest mitigations or alternatives to improve the contract's balance and enforceability.",
   'rate': 0.8},
  'financial_an